# C1-ml-fundamentals — Practice p20 — Solution

An assembly line measures one vibration reading per motor; faulty motors
(class `1`) tend to read higher. Work through the full workflow — later parts
consume earlier parts' results.

**(a)** The data cell below builds 90 labeled readings, already shuffled with
a seeded permutation. Split them: first 60 as `X_train, y_train`, last 30 as
`X_test, y_test` (exact names).

**(b)** Choose a cutoff `t` as in the lesson: among midpoints of consecutive
sorted *training* readings, keep the one with the highest training accuracy
for the rule "flag when reading ≥ t". Compute `train_acc` and `test_acc`
(exact names) for that cutoff and state in a comment whether the train-test
gap looks like overfitting.

**(c)** On the test set only, compute `test_precision`, `test_recall`, and
`test_f1` (exact names) for the chosen cutoff.

**(d)** *Reasoning is required.* In the markdown cell: the plant manager only
cares about not shipping faulty motors. Which single metric from (c) answers
her concern, what value did it take, and is the (b) gap evidence of
overfitting?

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

In [ ]:
SEED = 20260804
data_rng = np.random.default_rng(SEED)
readings_ok    = data_rng.normal(3.0, 1.0, size=60)
readings_fault = data_rng.normal(5.5, 1.0, size=30)

X = np.concatenate([readings_ok, readings_fault])
y = np.concatenate([np.zeros(60, dtype=int), np.ones(30, dtype=int)])

order = data_rng.permutation(90)
X, y = X[order], y[order]

In [ ]:
# (a) the split -- data is already shuffled, so slicing deals fairly
X_train, y_train = X[:60], y[:60]
X_test,  y_test  = X[60:], y[60:]
print(X_train.shape, X_test.shape)

In [ ]:
# (b) choose the cutoff on TRAINING data only
s = np.sort(X_train)
candidates = (s[:-1] + s[1:]) / 2
best_t, best_acc = float(candidates[0]), -1.0
for t in candidates:
    acc = np.mean((X_train >= t).astype(int) == y_train)
    if acc > best_acc:
        best_t, best_acc = float(t), float(acc)
t = best_t

train_acc = float(np.mean((X_train >= t).astype(int) == y_train))
test_acc  = float(np.mean((X_test  >= t).astype(int) == y_test))
print(f"t = {t:.3f}  train_acc = {train_acc:.3f}  test_acc = {test_acc:.3f}")
# The gap is small -- this simple rule is not overfitting.

In [ ]:
# (c) error-type metrics on the held-out test set
flag = X_test >= t
TP = np.sum(flag & (y_test == 1))
FP = np.sum(flag & (y_test == 0))
FN = np.sum(~flag & (y_test == 1))
test_precision = TP / (TP + FP)
test_recall    = TP / (TP + FN)
test_f1        = 2 * test_precision * test_recall / (test_precision + test_recall)
print(f"precision = {test_precision:.3f}  recall = {test_recall:.3f}  F1 = {test_f1:.3f}")

**(d)** Not shipping faulty motors means catching as many true faults as
possible — that is **recall**, which answers "of the actual faulty motors,
what fraction did we flag?". Here test recall is 0.857, so about
86% of faulty motors in the held-out set are caught; the manager
should focus on driving the missed fraction down, accepting some false
alarms. The train-test gap (0.983 vs 0.900) is
small, so the rule is not overfitting — a single threshold has too little
flexibility to memorize the training data.

### Answer check

In [ ]:
assert X_train.shape == (60,) and X_test.shape == (30,)
assert np.isclose(t, 4.493183731395844, atol=1e-9, rtol=0)
assert np.isclose(train_acc, 0.9833333333333333, atol=1e-9, rtol=0)
assert np.isclose(test_acc, 0.9, atol=1e-9, rtol=0)
assert np.isclose(test_precision, 0.9230769230769231, atol=1e-9, rtol=0)
assert np.isclose(test_recall, 0.8571428571428571, atol=1e-9, rtol=0)
assert np.isclose(test_f1, 0.888888888888889, atol=1e-9, rtol=0)
assert abs(train_acc - test_acc) < 0.15   # no overfitting gap
print("p20 OK")